# MorphSnap ML Training (Google Colab)
This notebook trains the MorphSnap parameter prediction model using a dataset generated locally.

### Instructions
1. In your Google Drive, create a folder named `MorphSnap`.
2. Upload your generated `dataset_manifest.json` into that `MorphSnap` folder.
3. Go to `Runtime > Change runtime type` and select **T4 GPU** to make training fast!
4. Click `Runtime > Run all` to execute the code below.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install missing dependencies
!pip install torch numpy

In [ ]:
# 3. Define the Model and Dataset Logic
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import json
import numpy as np
import os

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class MorphSnapDataset(Dataset):
    def __init__(self, metadata_path):
        with open(metadata_path, 'r') as f:
            self.data = json.load(f)
        
        self.samples = []
        # The generator extracts MFCC features and saves them in the JSON
        for item in self.data:
            features = np.array(item['audio_features'], dtype=np.float32)
            
            # Extract parameters
            params = []
            # Assumes the JSON stores the actual values, adjust if it stores dicts
            for p_name in sorted(item['parameters'].keys()):
                 val = item['parameters'][p_name]
                 params.append(val if isinstance(val, (int, float)) else val.get('value', 0.0))
            
            self.samples.append({
                'features': torch.tensor(features),
                'parameters': torch.tensor(params, dtype=np.float32)
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]['features'], self.samples[idx]['parameters']

class ParameterPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super(ParameterPredictor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_size),
            nn.Sigmoid() # Assuming parameters are normalized 0-1
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# 4. Training Loop
DRIVE_FOLDER = '/content/drive/My Drive/MorphSnap'
DATASET_FILE = os.path.join(DRIVE_FOLDER, 'dataset_manifest.json')
MODEL_OUTPUT = os.path.join(DRIVE_FOLDER, 'morphsnap_model.pth')

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(f"Could not find dataset at {DATASET_FILE}. Please upload it!")

print("Loading dataset...")
dataset = MorphSnapDataset(DATASET_FILE)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Get dimensions from first sample
sample_features, sample_params = dataset[0]
input_size = len(sample_features)
output_size = len(sample_params)

print(f"Initializing model (Input: {input_size}, Output: {output_size})...")
model = ParameterPredictor(input_size, output_size).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
print("Starting training loop...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for features, targets in dataloader:
        features, targets = features.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

print("Training complete! Saving model...")
torch.save(model.state_dict(), MODEL_OUTPUT)
print(f"Success: Model saved to {MODEL_OUTPUT}")